# Brent · `site_number` (web: "Site number")

**Goal:** understand the data, so that a chat bot can answer any question about it
more easily and correctly.

Brent only (golden rule): every query carries the location-based Brent filter.
Data comes from the **live API**.

Workflow: add a step here → discuss the findings → decide whether it belongs in
`knowledge/brent.md` → add it.

In [1]:
from pathlib import Path

import httpx
import pandas as pd
from dotenv import dotenv_values

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
env = dotenv_values(PROJECT_ROOT / ".env")

BASE_URL = "https://planningdata.london.gov.uk/api-guest"
HEADERS = {"X-API-AllowRequest": env["DATAHUB_ALLOW_REQUEST"]}


def brent_filter() -> dict:
    # Location-based scope: borough is Brent, or borough is missing and Brent decided it.
    return {
        "bool": {
            "should": [
                {"term": {"borough.raw": "Brent"}},
                {"bool": {
                    "filter": [{"term": {"lpa_name.raw": "Brent"}}],
                    "must_not": [{"exists": {"field": "borough.raw"}}],
                }},
            ],
            "minimum_should_match": 1,
        }
    }


def search(body: dict) -> dict:
    response = httpx.post(f"{BASE_URL}/applications/_search", headers=HEADERS, json=body, timeout=60)
    response.raise_for_status()
    return response.json()

## Step 1 · Nulls and empty values

Fetch `site_number` for every Brent application (paged with `search_after`, sorted by
`id`, because there are more than 10,000). Then count:
- **null**: no value at all;
- **empty or blank**: a string with nothing in it, or only whitespace.

In [2]:
rows, after = [], None
while True:
    body = {"size": 5000, "query": brent_filter(), "_source": ["id", "site_number"], "sort": [{"id": "asc"}]}
    if after:
        body["search_after"] = after
    page = search(body)["hits"]["hits"]
    if not page:
        break
    rows += [h["_source"] for h in page]
    after = page[-1]["sort"]

df = pd.DataFrame(rows).reindex(columns=["id", "site_number"])
print("Brent applications:", len(df))

Brent applications: 36952


In [3]:
value = df["site_number"]
is_null = value.isna()
is_blank = value.notna() & (value.astype(str).str.strip() == "")

summary = pd.Series({
    "null": int(is_null.sum()),
    "empty or blank": int(is_blank.sum()),
    "filled": int((~is_null & ~is_blank).sum()),
})
summary["total"] = summary.sum()
summary.to_frame("count").assign(**{"share_%": lambda t: (100 * t["count"] / len(df)).round(2)})

,count,share_%
null,31202,84.44
empty or blank,0,0.00
filled,5750,15.56
total,36952,100.00


### Findings · Step 1

Live API, 2026-11-09, 36,952 Brent applications:

| | Count | Share |
|---|---|---|
| null | 31,202 | 84.44% |
| empty or blank | 0 | 0% |
| filled | 5,750 | 15.56% |

- **Mostly empty:** only 15.6% of records have a site number.
- No blank values. `site_name` (Step 3 of its notebook) often already holds the house
  number as part of a full address, which may be why this field is rarely used.

## Step 2 · Most common site numbers

Count identical site numbers in `df` (exact text) and show the 30 most common.
`shown_as` uses `repr()` so hidden characters are visible.

In [4]:
print("distinct site numbers:", df["site_number"].nunique(), "| filled:", df["site_number"].notna().sum())
top = df["site_number"].value_counts().head(30).to_frame("count").reset_index()
top["shown_as"] = top["site_number"].map(repr)
top

distinct site numbers: 2753 | filled: 5750


,site_number,count,shown_as
0,2,52,'2'
1,2,43,2
2,1,39,1
3,1,37,'1'
4,8,36,8
5,14,31,14
6,3,30,'3'
7,10,29,10
8,18,28,'18'
9,19,28,19


### Findings · Step 2

Live API, 2026-11-09: 5,750 filled site numbers, 2,753 distinct.

- **Mostly plain house numbers** (1, 2, 3, 8, 10, 14 ...), each used 20-52 times.
- **Two JSON types for the same value:** `'2'` (string, 52) and `2` (number, 43) are both
  in the top 30, and the same for 1, 3, 7, 10, 12, 13. The schema says `string`. Exact
  matching on the raw values would treat `2` and `'2'` as different.
- The odd values seen in `site_name` Step 2 (`291-`, `19-`, `0`, `Unit 3,4 & 5`) are not
  in the top 30.

## Step 3 · Types of the filled values

The schema gives `site_number` the type `string`. For the filled values in `df`:
1. count the Python type (`str` = JSON string, `int` / `float` = JSON number);
2. split the strings into plain whole numbers (digits only) and everything else;
3. show the most common strings that are not plain numbers.

In [5]:
filled = df.loc[df["site_number"].notna(), ["id", "site_number"]].copy()
filled["type"] = filled["site_number"].map(lambda v: type(v).__name__)
display(filled["type"].value_counts().to_frame("count"))

strings = filled[filled["type"] == "str"].copy()
strings["digits_only"] = strings["site_number"].str.fullmatch(r"\d+")
display(strings["digits_only"].value_counts().rename({True: "digits only", False: "other text"}).to_frame("count"))

numbers = filled[filled["type"] != "str"]
print("numbers that are not whole:", int((numbers["site_number"] % 1 != 0).sum()))

other_text = strings[~strings["digits_only"]]
print("distinct other text:", other_text["site_number"].nunique())
other_text["site_number"].value_counts().head(30).to_frame("count")

,count
type,
str,4126
int,1624


,count
digits_only,
other text,2525
digits only,1601


numbers that are not whole: 0
distinct other text: 2154


,count
site_number,
1A,8
2 & 2A,7
2A,6
York House,6
"Valiant House, 365",5
"LONSDALE HOUSE, 560",5
85-91,5
9 & 9A,4
2B,4


### Findings · Step 3

Live API, 2026-11-09, the 5,750 filled site numbers:

| Kind | Count |
|---|---|
| JSON number (`int`), all whole numbers | 1,624 |
| string, digits only (e.g. `'2'`) | 1,601 |
| string, other text | 2,525 |

- **Not a number field.** 44% of filled values are text that isn't a plain number.
  `site_number` holds a mix of:
  - house numbers with letters: `1A`, `2B`, `24A`;
  - ranges and lists: `85-91`, `2-4`, `2 & 2A`, `85 & 87`;
  - building or site names: `York House`, `The Stadium`, `Northwick Park Hospital`,
    `CAR PARK`, `Garages`;
  - a building name with a number: `Valiant House, 365`, `LONSDALE HOUSE, 560`.
- **3,225 are plain whole numbers**, split between JSON numbers (1,624) and digit strings
  (1,601). The schema says `string`, so the numbers don't match it; loaded as text, `2` and
  `'2'` become the same value.
- Upper and lower case are mixed (`LONSDALE HOUSE` / `York House`).